# 04 — Statistical Analysis

**Goal:** put formal hypothesis tests behind the strongest EDA findings, with the right tests,
stated assumptions, and — critically — **effect sizes**, not just p-values.

**Why effect sizes lead here:** with n = 7,043, even trivial differences produce tiny p-values.
A p-value answers "could this be chance?"; an effect size answers "is this big enough to care
about?". A portfolio project that only reports p < 0.001 five times demonstrates the opposite of
statistical maturity.

**Test selection:**

| Relationship | Test | Why this test |
|---|---|---|
| Churn × categorical (Contract, InternetService, PaymentMethod, TechSupport, gender) | χ² test of independence | Two categorical variables; large expected counts (assumption verified below) |
| Numeric × churn (MonthlyCharges, tenure) | Mann-Whitney U | Distributions are visibly non-normal (U-shaped tenure, bimodal charges), so we avoid the t-test's normality assumption; MWU compares distributions without it |
| Effect sizes | Cramér's V (χ²), rank-biserial r (MWU) | Scale-free magnitude, comparable across tests |

Significance level **α = 0.05** throughout. We run 7 planned tests; with a Bonferroni correction
the working threshold is α/7 ≈ **0.007** — noted per test, though (spoiler) every non-control
p-value lands far below even that.

**Standing caution:** none of these tests establish causation. They quantify *association* in
observational, snapshot data.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats

from src.config import CLEAN_DATA_FILE

df = pd.read_csv(CLEAN_DATA_FILE)
df["churn01"] = (df["Churn"] == "Yes").astype(int)

ALPHA = 0.05
N_TESTS = 7
ALPHA_BONF = ALPHA / N_TESTS
results = []  # collected for the summary table
print(f"n = {len(df):,} | alpha = {ALPHA} | Bonferroni-adjusted alpha = {ALPHA_BONF:.4f}")

n = 7,043 | alpha = 0.05 | Bonferroni-adjusted alpha = 0.0071


## 1. Helper: chi-square test with assumption check and effect size

Cramér's V rescales χ² to [0, 1] so tests with different table sizes are comparable.
Interpretation guide (Cohen-style, for df* = min(r,c) − 1 = 1): ~0.10 small, ~0.30 medium,
~0.50 large. The helper also verifies the χ² validity condition (all expected counts ≥ 5).

In [2]:
def chi_square_report(col, label=None):
    """Chi-square test of independence between `col` and Churn, with Cramér's V."""
    table = pd.crosstab(df[col], df["Churn"])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
    ok = (expected >= 5).all()

    print(f"H0: Churn is independent of {col}")
    print(f"H1: Churn is associated with {col}")
    print(f"Test: chi-square test of independence ({table.shape[0]}x{table.shape[1]} table)")
    print(f"Assumption (all expected counts >= 5): {'PASS' if ok else 'FAIL'} "
          f"(min expected = {expected.min():.1f})")
    print(f"chi2 = {chi2:,.1f} | dof = {dof} | p = {p:.3g}")
    reject = p < ALPHA
    print(f"Decision at alpha={ALPHA}: {'REJECT H0' if reject else 'FAIL TO REJECT H0'}"
          f"{' (also below Bonferroni ' + format(ALPHA_BONF, '.4f') + ')' if p < ALPHA_BONF else ''}")
    print(f"Effect size (Cramer's V) = {cramers_v:.3f}")
    results.append({"relationship": f"Churn x {label or col}", "test": "chi-square",
                    "statistic": round(chi2, 1), "p": p, "effect": round(cramers_v, 3),
                    "effect_name": "Cramer's V"})
    return table

## 2. Churn × Contract

The strongest EDA association (42.7% → 2.8% churn across contract types).

In [3]:
table = chi_square_report("Contract")
table.assign(churn_rate=lambda t: (t["Yes"] / t.sum(axis=1) * 100).round(1))

H0: Churn is independent of Contract
H1: Churn is associated with Contract
Test: chi-square test of independence (3x2 table)
Assumption (all expected counts >= 5): PASS (min expected = 390.9)
chi2 = 1,184.6 | dof = 2 | p = 5.86e-258
Decision at alpha=0.05: REJECT H0 (also below Bonferroni 0.0071)
Effect size (Cramer's V) = 0.410


Churn,No,Yes,churn_rate
Contract,,,
Month-to-month,2220,1655,42.7
One year,1307,166,11.3
Two year,1647,48,2.8


**Interpretation:** the association is statistically decisive (p ≈ 6×10⁻²⁵⁸) and — more
importantly — **large in practical terms** (V = 0.41, the biggest effect in this analysis).
Based on the observed association, contract type is the dataset's dominant churn correlate.
We do not conclude that contracts *cause* retention: customers self-select into contracts,
and tenure is entangled with contract choice (notebook 03).

## 3. Churn × InternetService

In [4]:
table = chi_square_report("InternetService")
table.assign(churn_rate=lambda t: (t["Yes"] / t.sum(axis=1) * 100).round(1))

H0: Churn is independent of InternetService
H1: Churn is associated with InternetService
Test: chi-square test of independence (3x2 table)
Assumption (all expected counts >= 5): PASS (min expected = 405.0)
chi2 = 732.3 | dof = 2 | p = 9.57e-160
Decision at alpha=0.05: REJECT H0 (also below Bonferroni 0.0071)
Effect size (Cramer's V) = 0.322


Churn,No,Yes,churn_rate
InternetService,,,
DSL,1962,459,19.0
Fiber optic,1799,1297,41.9
No,1413,113,7.4


**Interpretation:** a solid medium effect (V = 0.32). Fiber's 41.9% churn versus DSL's 19.0%
is not sampling noise. The fiber–price confound from notebook 03 still applies: this test cannot
say whether the association reflects the product, its price point, or the customers it attracts.

## 4. Churn × PaymentMethod

In [5]:
table = chi_square_report("PaymentMethod")
table.assign(churn_rate=lambda t: (t["Yes"] / t.sum(axis=1) * 100).round(1))

H0: Churn is independent of PaymentMethod
H1: Churn is associated with PaymentMethod
Test: chi-square test of independence (4x2 table)
Assumption (all expected counts >= 5): PASS (min expected = 403.9)
chi2 = 648.1 | dof = 3 | p = 3.68e-140
Decision at alpha=0.05: REJECT H0 (also below Bonferroni 0.0071)
Effect size (Cramer's V) = 0.303


Churn,No,Yes,churn_rate
PaymentMethod,,,
Bank transfer (automatic),1286,258,16.7
Credit card (automatic),1290,232,15.2
Electronic check,1294,1071,45.3
Mailed check,1304,308,19.1


**Interpretation:** medium effect (V = 0.30). The 4-level table hides the real structure —
both automatic methods sit at 15–17% churn while electronic check sits at 45.3% — which is why
feature engineering (notebook 05) will encode *manual vs automatic* rather than lean on the raw
4-level variable alone.

## 5. Churn × TechSupport (internet customers only)

Restricted to internet customers (n = 5,517) so the structural "No internet service" level
cannot inflate the effect — the honest version of this test.

In [6]:
internet = df[df["InternetService"] != "No"]
table = pd.crosstab(internet["TechSupport"], internet["Churn"])
chi2, p, dof, expected = stats.chi2_contingency(table)
n = table.values.sum()
v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
print("H0: among internet customers, Churn is independent of TechSupport")
print("H1: among internet customers, Churn is associated with TechSupport")
print(f"Test: chi-square ({table.shape[0]}x{table.shape[1]}); min expected = {expected.min():.1f} (>=5 PASS)")
print(f"chi2 = {chi2:,.1f} | dof = {dof} | p = {p:.3g}")
print(f"Decision at alpha={ALPHA}: {'REJECT H0' if p < ALPHA else 'FAIL TO REJECT H0'}")
print(f"Effect size (Cramer's V) = {v:.3f}")
results.append({"relationship": "Churn x TechSupport (internet only)", "test": "chi-square",
                "statistic": round(chi2, 1), "p": p, "effect": round(v, 3),
                "effect_name": "Cramer's V"})
table.assign(churn_rate=lambda t: (t["Yes"] / t.sum(axis=1) * 100).round(1))

H0: among internet customers, Churn is independent of TechSupport
H1: among internet customers, Churn is associated with TechSupport
Test: chi-square (2x2); min expected = 650.6 (>=5 PASS)
chi2 = 414.3 | dof = 1 | p = 4.35e-92
Decision at alpha=0.05: REJECT H0
Effect size (Cramer's V) = 0.274


Churn,No,Yes,churn_rate
TechSupport,,,
No,2027,1446,41.6
Yes,1734,310,15.2


**Interpretation:** even after removing the structural level, the association holds with a
small-to-medium effect (V = 0.27). Customers without tech support churn at 41.6% vs 15.2% with
it. Plausibly a mix of real protection and selection (engaged customers buy support); the test
cannot separate the two.

## 6. Negative control: Churn × gender

EDA showed 26.9% vs 26.2% — this test *should* fail to reject. Running it anyway is the
statistical honesty check: if our pipeline found "significance" here, we'd suspect our method.

In [7]:
table = chi_square_report("gender")
table.assign(churn_rate=lambda t: (t["Yes"] / t.sum(axis=1) * 100).round(1))

H0: Churn is independent of gender
H1: Churn is associated with gender
Test: chi-square test of independence (2x2 table)
Assumption (all expected counts >= 5): PASS (min expected = 925.6)
chi2 = 0.5 | dof = 1 | p = 0.487
Decision at alpha=0.05: FAIL TO REJECT H0
Effect size (Cramer's V) = 0.008


Churn,No,Yes,churn_rate
gender,,,
Female,2549,939,26.9
Male,2625,930,26.2


**Interpretation:** p = 0.49 — no evidence of association, effect size ≈ 0.008 (nil). Gender
does not distinguish churners, confirming the EDA read and pre-registering our expectation that
it ranks near zero in model importances. **Absence of evidence here is informative** because with
n = 7,043 this test had power to detect even a ~3-point rate difference.

## 7. Do churned customers pay different monthly charges? (Mann-Whitney U)

**Assumption check first:** the t-test assumes approximate normality within groups. Monthly
charges are bimodal (the ~\$20 basic-plan mass plus a broad \$60–110 band) and tenure is
U-shaped — normality clearly fails *by inspection* (with n this large, formal normality tests
reject everything, so shape inspection is the meaningful check; skewness/kurtosis shown below).
Mann-Whitney U compares the two distributions without that assumption.

MWU's H0: the two distributions are identical — equivalently, a randomly drawn churner's bill is
equally likely to be higher or lower than a retained customer's. The rank-biserial correlation
r = 2·AUC − 1 (where AUC = P(churner > retained)) gives the effect size.

In [8]:
def mann_whitney_report(col):
    """Mann-Whitney U comparing `col` across churn groups + rank-biserial effect size."""
    yes = df.loc[df["Churn"] == "Yes", col]
    no = df.loc[df["Churn"] == "No", col]
    print(f"Group shapes: churned skew={yes.skew():.2f}, retained skew={no.skew():.2f} "
          f"(and see notebook 03 histograms: non-normal by inspection)")
    u, p = stats.mannwhitneyu(yes, no, alternative="two-sided")
    auc = u / (len(yes) * len(no))       # P(churner value > retained value)
    rank_biserial = 2 * auc - 1
    print(f"H0: distribution of {col} is identical for churned and retained customers")
    print(f"H1: the distributions differ")
    print(f"Test: Mann-Whitney U (two-sided) | U = {u:,.0f} | p = {p:.3g}")
    reject = p < ALPHA
    print(f"Decision at alpha={ALPHA}: {'REJECT H0' if reject else 'FAIL TO REJECT H0'}"
          f"{' (also below Bonferroni)' if p < ALPHA_BONF else ''}")
    print(f"P({col} of churner > retained) = {auc:.3f} | rank-biserial r = {rank_biserial:+.3f}")
    print(f"Medians: churned = {yes.median():,.2f} | retained = {no.median():,.2f}")
    results.append({"relationship": f"{col} x Churn", "test": "Mann-Whitney U",
                    "statistic": round(u, 0), "p": p, "effect": round(rank_biserial, 3),
                    "effect_name": "rank-biserial r"})

mann_whitney_report("MonthlyCharges")

Group shapes: churned skew=-0.73, retained skew=-0.03 (and see notebook 03 histograms: non-normal by inspection)
H0: distribution of MonthlyCharges is identical for churned and retained customers
H1: the distributions differ
Test: Mann-Whitney U (two-sided) | U = 6,003,126 | p = 3.31e-54
Decision at alpha=0.05: REJECT H0 (also below Bonferroni)
P(MonthlyCharges of churner > retained) = 0.621 | rank-biserial r = +0.242
Medians: churned = 79.65 | retained = 64.43


**Interpretation:** churners' bills are stochastically higher — a randomly chosen churner
out-pays a randomly chosen retained customer **62.1%** of the time (r = +0.24, small).
Medians \$79.65 vs \$64.43. Real, but far from deterministic — plenty of cheap-plan churners and
expensive loyalists exist.

## 8. Does tenure differ between churned and retained customers?

In [9]:
mann_whitney_report("tenure")

Group shapes: churned skew=1.15, retained skew=-0.03 (and see notebook 03 histograms: non-normal by inspection)
H0: distribution of tenure is identical for churned and retained customers
H1: the distributions differ
Test: Mann-Whitney U (two-sided) | U = 2,515,538 | p = 2.42e-208
Decision at alpha=0.05: REJECT H0 (also below Bonferroni)
P(tenure of churner > retained) = 0.260 | rank-biserial r = -0.480
Medians: churned = 10.00 | retained = 38.00


**Interpretation:** the largest numeric effect: a random churner has *shorter* tenure than a
random retained customer **74.0%** of the time (r = −0.48, medium-to-large; medians 10 vs 38
months). Formal confirmation that churn is an early-lifecycle phenomenon.

**Causal-direction caution — this one is subtle:** short tenure doesn't just "predict" churn;
churning *truncates* tenure. Anyone who leaves early necessarily has short tenure, so part of
this association is mechanical. This matters for interpretation later: tenure is a legitimate
*predictor* (it's known at scoring time) but reading its model importance as "staying longer
causes loyalty" would be circular.

## 9. Summary of all tests

In [10]:
summary = pd.DataFrame(results)
summary["p"] = summary["p"].map(lambda v: f"{v:.3g}")
summary["significant (alpha=0.05)"] = summary["p"].astype(float) < ALPHA
summary

,relationship,test,statistic,p,effect,effect_name,significant (alpha=0.05)
0,Churn x Contract,chi-square,1184.6,5.86e-258,0.410,Cramer's V,True
1,Churn x InternetService,chi-square,732.3,9.57e-160,0.322,Cramer's V,True
2,Churn x PaymentMethod,chi-square,648.1,3.68e-140,0.303,Cramer's V,True
3,Churn x TechSupport (internet only),chi-square,414.3,4.35e-92,0.274,Cramer's V,True
4,Churn x gender,chi-square,0.5,0.487,0.008,Cramer's V,False
5,MonthlyCharges x Churn,Mann-Whitney U,6003126.0,3.31e-54,0.242,rank-biserial r,True
6,tenure x Churn,Mann-Whitney U,2515538.0,2.42e-208,-0.480,rank-biserial r,True


## Conclusions

1. **Every substantive association from EDA survives formal testing** — contract (V = 0.41,
   large), internet service (V = 0.32, medium), payment method (V = 0.30, medium), tech support
   among internet customers (V = 0.27), monthly charges (r = +0.24), tenure (r = −0.48). All
   p-values sit far below the Bonferroni-adjusted threshold, so multiple testing does not
   threaten any conclusion.
2. **The negative control behaved** — gender: p = 0.49, V ≈ 0.008. The pipeline doesn't
   manufacture significance, and we have a pre-registered expectation for the modeling phase.
3. **Effect-size hierarchy** (contract > tenure > internet ≈ payment ≈ charges ≫ gender) gives
   us an evidence-based prior for what feature importance *should* look like. If the models
   disagree wildly, that's a prompt to investigate, not to celebrate novelty.
4. **Nothing here is causal.** Snapshot, observational, self-selected treatment groups — the
   tests quantify associations; interventions built on them are hypotheses to A/B test.
